In [46]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pymap3d as pm
import math
import seaborn as sns
import datetime as dt
import time
import os
import glob
from pathlib import Path

# ========================
# КОНФИГУРАЦИЯ
# ========================

# SOLUTIONS_DIR = "solutions_gps_AR"  # Папка с вашими POS-файлами
# SOLUTIONS_DIR = "solutions_gps_NOAR"  # Папка с вашими POS-файлами
# SOLUTIONS_DIR = "solutions_all_NOAR"  # Папка с вашими POS-файлами
SOLUTIONS_DIR = "solutions_all_AR"  # Папка с вашими POS-файлами
# ========================
# КОНФИГУРАЦИЯ
# ========================
REF_POINT = {'x': 54.9872753361111, 'y': 82.864814275, 'z': 109.647}  # Эталонные координаты

# ========================
# НАСТРОЙКИ ЧТЕНИЯ ФАЙЛОВ
# ========================
USE_FULL_FILE = True      # Если True — читать весь файл
USE_LAST_N_ROWS = 0       # Если > 0 — читать только последние N строк (игнорируется, если USE_FULL_FILE=True)
# ========================
# СТАТИСТИКА ПО ПОСЛЕДНИМ РЕШЕНИЯМ
# ========================
STAT_LAST_N = 10          # Сколько последних решений использовать для статистики
ENABLE_LAST_N_STATS = True  # Включить/выключить расчёт

# ========================
# НАСТРОЙКИ ГРАФИКА СКЛЕЕННЫХ ДАННЫХ
# ========================
MARK_SOLUTION_INDEX = 1000      # Например: 1500 — красная линия на 1500-м решении (глобальный индекс)
# MARK_SOLUTION_INDEX = -1      # Последнее решение
# MARK_SOLUTION_INDEX = None    # Не рисовать
# GPSAR
# SESSION_MARK_CONFIG = {
#     "sol_000.pos": 0.8,        # 30% от длины
#     "sol_001.pos": "None",        # 50%
#     "sol_002.pos": "None",      # конец
#     "sol_003.pos": 0.45, # последний фикс
#     "sol_004.pos": "None",  
#     "sol_005.pos": 0.2,
# }
# GPSNOAR
# SESSION_MARK_CONFIG = {
#     "sol_000.pos": 0.5,        # 30% от длины
#     "sol_001.pos": "None",        # 50%
#     "sol_002.pos": "None",      # конец
#     "sol_003.pos": "None", # последний фикс
#     "sol_004.pos": "None",  
#     "sol_005.pos": 0.2,
# }

# ALLNOAR
# SESSION_MARK_CONFIG = {
#     "sol_000.pos": 0.14,        # 30% от длины
#     "sol_001.pos": 0.2,        # 50%
#     "sol_002.pos": "None",      # конец
#     "sol_003.pos": 0.2, # последний фикс
#     "sol_004.pos": 0.7,  
#     "sol_005.pos": "None",
# }
# ALLAR
# SESSION_MARK_CONFIG = {
#     "sol_000.pos": 0.29,        # 30% от длины
#     "sol_001.pos": "None",        # 50%
#     "sol_002.pos": "None",      # конец
#     "sol_003.pos": 0.21, # последний фикс
#     "sol_004.pos": 0.15,  
#     "sol_005.pos": 0.3,
# }


SESSION_MARK_CONFIG = {
    "sol_000.pos": 0.1,        # 30% от длины
    "sol_001.pos": 0.1,        # 50%
    "sol_002.pos": 0.1,      # конец
    "sol_003.pos": 0.1, # последний фикс
    "sol_004.pos": 0.1,  
    "sol_005.pos": 0.1,
}
# ========================
# СТАТИСТИКА ОТ МЕТКИ ДО КОНЦА СЕАНСА
# ========================
ENABLE_STATS_FROM_MARK = True  # Включить/выключить расчёт
SESSION_LABELS_ON_XAXIS = True  # Подписывать сеансы под осью X (True/False)

# ========================
# ФИЛЬТРЫ ПО ОШИБКАМ ENU
# ========================
MAX_HEIGHT_ERROR = 3000      # Максимальное допустимое отклонение высоты в метрах
MAX_EAST_ERROR = 100      # Максимальное допустимое отклонение по востоку (E)
MAX_NORTH_ERROR = 1000       # Максимальное допустимое отклонение по северу (N)
MAX_HORIZONTAL_ERROR = 15000  # Максимальное допустимое горизонтальное отклонение (2D)
MAX_3D_ERROR = 200000        # Максимальное допустимое 3D отклонение
IQR_FACTOR = 1.5 
# ========================
# СМЕЩЕНИЕ (BIAS) ДЛЯ КООРДИНАТ ENU — ПРИМЕНЯЕТСЯ ПЕРЕД ВЫЧИСЛЕНИЕМ СТАТИСТИК
# ========================
# ENU_BIAS_E = 0.0    # Смещение по востоку (м) — например, +0.03
# ENU_BIAS_N = 0.1055    # Смещение по северу (м) — например, -0.01
# ENU_BIAS_U = 0.4187    # Смещение по высоте (м) — например, +0.05

# ENU_BIAS_E = 0.027    # Смещение по востоку (м) — например, +0.03
# ENU_BIAS_N = 0.0925    # Смещение по северу (м) — например, -0.01
# ENU_BIAS_U = 0.3607  

# ENU_BIAS_E = .027    # Смещение по востоку (м) — например, +0.03
# ENU_BIAS_N = .013    # Смещение по северу (м) — например, -0.01
# ENU_BIAS_U = .058  

ENU_BIAS_E = 0    # Смещение по востоку (м) — например, +0.03
ENU_BIAS_N = 0    # Смещение по северу (м) — например, -0.01
ENU_BIAS_U = 0  
# Фильтр по типу решений
ONLY_FIXED_SOLUTIONS = False  # Если True - использовать только фиксированные решения (quality=1)
# ========================
# ФУНКЦИИ ПРЕОБРАЗОВАНИЯ
# ========================

def reduction_to_enu(df, ref_point):
    """Преобразует координаты в систему ENU относительно эталона"""
    if len(df) == 0:
        return df
    
    enu = []
    for i in range(len(df)):
        try:
            e, n, u = pm.geodetic2enu(
                df['latitude(deg)'].iloc[i], 
                df['longitude(deg)'].iloc[i], 
                df['height(m)'].iloc[i], 
                ref_point['x'], ref_point['y'], ref_point['z']
            )
            enu.append([e, n, u])
        except Exception as e:
            enu.append([np.nan, np.nan, np.nan])
    
    enu_df = pd.DataFrame(enu, columns=['E', 'N', 'U'])
    result = pd.concat([df.reset_index(drop=True), enu_df], axis=1)
    
    # Удаляем строки с NaN в ENU координатах
    initial_count = len(result)
    result = result.dropna(subset=['E', 'N', 'U'])
    final_count = len(result)
    
    if initial_count != final_count:
        print(f"   Удалено {initial_count - final_count} строк с ошибками преобразования")
    
    return result

def filter_fixed_solutions(df_enu):
    """
    Фильтрует только фиксированные решения (quality = 1)
    """
    if len(df_enu) == 0:
        return df_enu
    
    if 'quality' not in df_enu.columns:
        return df_enu
    
    # Фильтруем только фиксированные решения
    df_fixed = df_enu[df_enu['quality'] == 1].copy()
    
    initial_count = len(df_enu)
    final_count = len(df_fixed)
    
    if initial_count > 0:
        print(f"   Фикс: {initial_count} → {final_count} решений (нефикс: {initial_count - final_count})")
    
    return df_fixed

def filter_solutions_by_enu_limits(df_enu, 
                                 max_east=MAX_EAST_ERROR,
                                 max_north=MAX_NORTH_ERROR, 
                                 max_height=MAX_HEIGHT_ERROR,
                                 max_horizontal=MAX_HORIZONTAL_ERROR,
                                 max_3d=MAX_3D_ERROR):
    """
    Фильтрует решения по всем заданным пределам ENU
    """
    if len(df_enu) == 0:
        return df_enu
    
    df_filtered = df_enu.copy()
    
    # Применяем фильтры последовательно
    filters_applied = []
    
    # Фильтр по востоку (E)
    if max_east is not None and 'E' in df_filtered.columns:
        initial_count = len(df_filtered)
        df_filtered = df_filtered[df_filtered['E'].abs() <= max_east]
        filtered_count = initial_count - len(df_filtered)
        if filtered_count > 0:
            filters_applied.append(f"E≤{max_east}м(-{filtered_count})")
    
    # Фильтр по северу (N)
    if max_north is not None and 'N' in df_filtered.columns:
        initial_count = len(df_filtered)
        df_filtered = df_filtered[df_filtered['N'].abs() <= max_north]
        filtered_count = initial_count - len(df_filtered)
        if filtered_count > 0:
            filters_applied.append(f"N≤{max_north}м(-{filtered_count})")
    
    # Фильтр по высоте (U)
    if max_height is not None and 'U' in df_filtered.columns:
        initial_count = len(df_filtered)
        df_filtered = df_filtered[df_filtered['U'].abs() <= max_height]
        filtered_count = initial_count - len(df_filtered)
        if filtered_count > 0:
            filters_applied.append(f"U≤{max_height}м(-{filtered_count})")
    
    # Фильтр по горизонтальной ошибке (2D)
    if max_horizontal is not None and all(col in df_filtered.columns for col in ['E', 'N']):
        initial_count = len(df_filtered)
        horizontal_errors = np.sqrt(df_filtered['E']**2 + df_filtered['N']**2)
        df_filtered = df_filtered[horizontal_errors <= max_horizontal]
        filtered_count = initial_count - len(df_filtered)
        if filtered_count > 0:
            filters_applied.append(f"2D≤{max_horizontal}м(-{filtered_count})")
    
    # Фильтр по 3D ошибке
    if max_3d is not None and all(col in df_filtered.columns for col in ['E', 'N', 'U']):
        initial_count = len(df_filtered)
        errors_3d = np.sqrt(df_filtered['E']**2 + df_filtered['N']**2 + df_filtered['U']**2)
        df_filtered = df_filtered[errors_3d <= max_3d]
        filtered_count = initial_count - len(df_filtered)
        if filtered_count > 0:
            filters_applied.append(f"3D≤{max_3d}м(-{filtered_count})")
    
    # Выводим информацию о примененных фильтрах
    if filters_applied:
        print(f"   Фильтры: {', '.join(filters_applied)}")
    
    initial_total = len(df_enu)
    final_total = len(df_filtered)
    
    if initial_total > 0:
        print(f"   Итог: {initial_total} → {final_total} решений")
    
    return df_filtered

def filter_outliers_by_iqr(df_enu, iqr_factor=1.5, components=['E', 'N', 'U']):
    """
    Фильтрует выбросы по методу IQR для указанных компонентов E, N, U.
    Удаляет строки, где хотя бы один компонент выходит за пределы [Q1 - 1.5*IQR, Q3 + 1.5*IQR].
    
    Параметры:
        df_enu: DataFrame с колонками E, N, U
        iqr_factor: коэффициент для IQR (по умолчанию 1.5 — стандартный)
        components: список компонентов для фильтрации
    
    Возвращает:
        Отфильтрованный DataFrame
    """
    if len(df_enu) == 0:
        return df_enu
    
    df_filtered = df_enu.copy()
    initial_count = len(df_filtered)
    filters_applied = []
    
    for comp in components:
        if comp not in df_filtered.columns:
            continue
            
        Q1 = df_filtered[comp].quantile(0.25)
        Q3 = df_filtered[comp].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - iqr_factor * IQR
        upper_bound = Q3 + iqr_factor * IQR
        
        # Сохраняем только строки, где значение в пределах
        mask = (df_filtered[comp] >= lower_bound) & (df_filtered[comp] <= upper_bound)
        df_filtered = df_filtered[mask]
        
        filtered_count = initial_count - len(df_filtered)
        if filtered_count > 0:
            filters_applied.append(f"{comp}-IQR(-{filtered_count})")
    
    if filters_applied:
        print(f"   🚫 Выбросы: {', '.join(filters_applied)}")
        print(f"   Итог: {initial_count} → {len(df_filtered)} решений после IQR")
    
    return df_filtered

def select_best_solutions(df_enu, sol_count=USE_LAST_N_ROWS):
    """
    Выбирает лучшие USE_LAST_N_ROWS решений по близости к эталону с учетом всех фильтров
    """
    if len(df_enu) == 0:
        return df_enu
    
    # Сначала применяем фильтр по типу решений (фикс/нефикс)
    if ONLY_FIXED_SOLUTIONS:
        df_filtered = filter_fixed_solutions(df_enu)
    else:
        df_filtered = df_enu.copy()
    
    if len(df_filtered) == 0:
        print("   ❌ Нет решений после фильтрации по типу")
        return df_filtered
    
    # Затем применяем фильтры по пределам ENU
    df_filtered = filter_solutions_by_enu_limits(
        df_filtered,
        max_east=MAX_EAST_ERROR,
        max_north=MAX_NORTH_ERROR,
        max_height=MAX_HEIGHT_ERROR,
        max_horizontal=MAX_HORIZONTAL_ERROR,
        max_3d=MAX_3D_ERROR
    )
    
    if len(df_filtered) == 0:
        print("   ❌ Нет решений после фильтрации по ENU пределам")
        return df_filtered
    
    # ✅ НОВЫЙ ШАГ: Фильтрация выбросов по IQR
    df_filtered = filter_outliers_by_iqr(df_filtered, iqr_factor=1.5, components=['E', 'N', 'U'])
    
    if len(df_filtered) == 0:
        print("   ❌ Нет решений после фильтрации выбросов (IQR)")
        return df_filtered
    
    # Вычисляем 3D расстояние для отфильтрованных решений
    df_filtered = df_filtered.copy()
    df_filtered['distance_3d'] = np.sqrt(
        df_filtered['E']**2 + 
        df_filtered['N']**2 + 
        df_filtered['U']**2
    )
    
    # Берем самые близкие решения из отфильтрованных
    if len(df_filtered) <= sol_count:
        best_solutions = df_filtered
    else:
        best_solutions = df_filtered.nsmallest(sol_count, 'distance_3d')
    
    solution_type = "ФИКСИРОВАННЫХ" if ONLY_FIXED_SOLUTIONS else "всех"
    print(f"   ✅ Отобрано {len(best_solutions)} {solution_type} решений")
    
    return best_solutions

# ========================
# ФУНКЦИИ ЧТЕНИЯ ФАЙЛОВ
# ========================

def read_pos_file_corrected(file_path):
    """
    Чтение POS-файла с поддержкой:
      - полного чтения (USE_FULL_FILE)
      - чтения последних N строк (USE_LAST_N_ROWS)
    """
    try:
        # Определяем, сколько строк читать
        if USE_FULL_FILE:
            nrows_to_read = None  # весь файл
            print(f"   📖 Чтение всего файла: {os.path.basename(file_path)}")
        elif USE_LAST_N_ROWS > 0:
            # Считаем общее число строк (без комментариев)
            with open(file_path, 'r') as f:
                total_lines = sum(1 for line in f if not line.strip().startswith('%'))
            skip_rows = max(0, total_lines - USE_LAST_N_ROWS)
            nrows_to_read = USE_LAST_N_ROWS
            print(f"   📖 Чтение последних {USE_LAST_N_ROWS} строк из {total_lines} (файл: {os.path.basename(file_path)})")
        else:
            nrows_to_read = None
            skip_rows = 0

        # Чтение данных
        if USE_FULL_FILE or USE_LAST_N_ROWS <= 0:
            df = pd.read_csv(
                file_path,
                sep=r'\s+',
                comment='%',
                header=None,
                names=['date', 'time', 'lat', 'lon', 'height', 'Q', 'ns',
                       'sde', 'sdn', 'sdu', 'sdne', 'sdeu', 'sdun', 'age', 'ratio'],
                engine='python',
                skipinitialspace=True,
                skiprows=1  # пропускаем первую строку (обычно заголовок после %)
            )
        else:
            # Читаем с пропуском начальных строк
            df = pd.read_csv(
                file_path,
                sep=r'\s+',
                comment='%',
                header=None,
                names=['date', 'time', 'lat', 'lon', 'height', 'Q', 'ns',
                       'sde', 'sdn', 'sdu', 'sdne', 'sdeu', 'sdun', 'age', 'ratio'],
                engine='python',
                skipinitialspace=True,
                skiprows=lambda x: x == 0 or x <= skip_rows,  # пропускаем заголовок и первые skip_rows строк данных
                nrows=nrows_to_read
            )

        if df.empty:
            print(f"   ⚠️  Файл {file_path} пуст после фильтрации")
            return df

        # Создаём временную метку
        try:
            df['UTC'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['time'].astype(str))
        except Exception as e:
            print(f"   ⚠️  Ошибка парсинга времени, генерация индекса: {e}")
            df['UTC'] = pd.date_range(start='2024-01-01', periods=len(df), freq='1S')

        # Стандартизация имён
        df = df.rename(columns={
            'lat': 'latitude(deg)',
            'lon': 'longitude(deg)',
            'height': 'height(m)',
            'Q': 'quality'
        })

        total_solutions = len(df)
        fixed_solutions = len(df[df['quality'] == 1])
        fix_percentage = (fixed_solutions / total_solutions * 100) if total_solutions > 0 else 0

        print(f"✅ Загружено: {total_solutions} записей (фикс: {fixed_solutions}, {fix_percentage:.1f}%)")
        return df

    except Exception as e:
        print(f"❌ Ошибка чтения {file_path}: {e}")
        return pd.DataFrame()

# ========================
# ФУНКЦИИ АНАЛИЗА
# ========================



def calculate_comprehensive_statistics(df, segment_name=""):
    if len(df) == 0:
        return None

    df_analysis = df.copy()

    # Применяем смещение ENU (если заданы)
    bias_applied = False
    if ENU_BIAS_E != 0.0 and 'E' in df_analysis.columns:
        df_analysis['E'] = df_analysis['E'] - ENU_BIAS_E
        bias_applied = True
    if ENU_BIAS_N != 0.0 and 'N' in df_analysis.columns:
        df_analysis['N'] = df_analysis['N'] - ENU_BIAS_N
        bias_applied = True
    if ENU_BIAS_U != 0.0 and 'U' in df_analysis.columns:
        df_analysis['U'] = df_analysis['U'] - ENU_BIAS_U
        bias_applied = True

    if bias_applied:
        print(f"   ⚙️  Применено смещение ENU: E={ENU_BIAS_E:+.4f}m, N={ENU_BIAS_N:+.4f}m, U={ENU_BIAS_U:+.4f}m")

    stats = {
        'segment': segment_name,
        'solutions_count': len(df_analysis),
        'fix_count': len(df_analysis) if ONLY_FIXED_SOLUTIONS else (df_analysis['quality'] == 1).sum(),
        'fix_percentage': 100.0 if ONLY_FIXED_SOLUTIONS else (df_analysis['quality'] == 1).sum() / len(df_analysis) * 100,
    }

    # === НОВОЕ: мин/макс количество спутников ===
    if 'ns' in df_analysis.columns:
        stats['ns_min'] = df_analysis['ns'].min()
        stats['ns_max'] = df_analysis['ns'].max()
        stats['mean_ns'] = df_analysis['ns'].mean()  # уже было, но оставим
    else:
        stats['ns_min'] = np.nan
        stats['ns_max'] = np.nan
        stats['mean_ns'] = np.nan

    # ... остальная статистика (высота, ENU, 2D/3D и т.д.) ...
    
    # Добавляем информацию о высотных ошибках (после смещения)
    if 'U' in df_analysis.columns:
        height_errors = df_analysis['U'].abs()
        stats['height_error_max'] = height_errors.max()
        stats['height_error_mean'] = height_errors.mean()
        stats['height_error_std'] = height_errors.std()
        stats['height_error_median'] = height_errors.median()
    
    # Добавляем информацию о 3D ошибках если есть
    if 'distance_3d' in df_analysis.columns:
        stats['3d_error_mean'] = df_analysis['distance_3d'].mean()
        stats['3d_error_std'] = df_analysis['distance_3d'].std()
        stats['3d_error_min'] = df_analysis['distance_3d'].min()
        stats['3d_error_max'] = df_analysis['distance_3d'].max()
        stats['3d_error_median'] = df_analysis['distance_3d'].median()
    
    # Статистика по ENU координатам (уже с поправкой)
    for component in ['E', 'N', 'U']:
        if component in df_analysis.columns:
            values = df_analysis[component].dropna()
            if len(values) > 0:
                # Основные статистические параметры
                stats[f'{component}_mean'] = values.mean()  # Среднее (систематическая ошибка ПОСЛЕ поправки)
                stats[f'{component}_std'] = values.std()    # СКО (случайная ошибка)
                stats[f'{component}_rms'] = np.sqrt(np.mean(values**2))  # СКП
                stats[f'{component}_min'] = values.min()
                stats[f'{component}_max'] = values.max()
                stats[f'{component}_median'] = values.median()
                stats[f'{component}_mad'] = (values - values.median()).abs().median()  # Median Absolute Deviation
    
    # 2D и 3D ошибки (после смещения)
    if all(col in df_analysis.columns for col in ['E', 'N']):
        E_vals = df_analysis['E'].dropna()
        N_vals = df_analysis['N'].dropna()
        if len(E_vals) > 0 and len(N_vals) > 0:
            horiz_errors = np.sqrt(E_vals**2 + N_vals**2)
            stats['2D_mean'] = horiz_errors.mean()
            stats['2D_std'] = horiz_errors.std()
            stats['2D_rms'] = np.sqrt(np.mean(horiz_errors**2))
            stats['2D_median'] = horiz_errors.median()
    
    if all(col in df_analysis.columns for col in ['E', 'N', 'U']):
        E_vals = df_analysis['E'].dropna()
        N_vals = df_analysis['N'].dropna()
        U_vals = df_analysis['U'].dropna()
        if len(E_vals) > 0 and len(N_vals) > 0 and len(U_vals) > 0:
            total_errors = np.sqrt(E_vals**2 + N_vals**2 + U_vals**2)
            stats['3D_mean'] = total_errors.mean()
            stats['3D_std'] = total_errors.std()
            stats['3D_rms'] = np.sqrt(np.mean(total_errors**2))
            stats['3D_median'] = total_errors.median()
    
    # Дополнительные метрики точности
    if all(col in stats for col in ['E_std', 'N_std', 'U_std']):
        stats['horizontal_precision'] = np.sqrt(stats['E_std']**2 + stats['N_std']**2)
        stats['vertical_precision'] = stats['U_std']
    
    return stats

def improved_main_analysis():
    """Улучшенная основная функция анализа с фильтрацией по ENU пределам"""
    
    # Загрузка всех решений
    print("📁 Загрузка POS-файлов...")
    solutions = {}
    pos_files = sorted(glob.glob(os.path.join(SOLUTIONS_DIR, "*.pos")))
    
    for file_path in pos_files:
        file_name = os.path.basename(file_path)
        df = read_pos_file_corrected(file_path)
        if len(df) > 0:
            solutions[file_name] = df
    
    if not solutions:
        print("❌ Не найдено POS-файлов для анализа")
        return None, None
    
    # Обработка каждого сегмента с фильтрацией
    solutions_stats = []
    processed_data = {}
    file_statistics = []
    
    print(f"\n🔄 Обработка {len(solutions)} сегментов с фильтрацией...")
    print(f"   Фильтры: {'ТОЛЬКО ФИКС' if ONLY_FIXED_SOLUTIONS else 'Все решения'}")
    print(f"   Пределы: E≤{MAX_EAST_ERROR}м, N≤{MAX_NORTH_ERROR}м, U≤{MAX_HEIGHT_ERROR}м")
    print(f"            2D≤{MAX_HORIZONTAL_ERROR}м, 3D≤{MAX_3D_ERROR}м")
    
    for file_name, df in solutions.items():
        print(f"\n🔍 Обработка {file_name}...")
        
        # Преобразование в ENU
        df_enu = reduction_to_enu(df.copy(), REF_POINT)
        
        if len(df_enu) > 0:
            # Статистика по файлу до фильтрации
            total_solutions = len(df_enu)
            fixed_solutions = len(df_enu[df_enu['quality'] == 1])
            
            file_statistics.append({
                'file_name': file_name,
                'total_solutions': total_solutions,
                'fixed_solutions': fixed_solutions,
                'fix_percentage': (fixed_solutions / total_solutions * 100) if total_solutions > 0 else 0
            })
            
            print(f"   Всего решений: {total_solutions}")
            print(f"   Фиксированных: {fixed_solutions} ({(fixed_solutions/total_solutions*100):.1f}%)")
            
            # Выбираем лучшие решения с учетом всех фильтров
            df_best = select_best_solutions(df_enu, USE_LAST_N_ROWS)
            
            if len(df_best) > 0:
                processed_data[file_name] = df_best
                
                # Расчет статистики
                stats = calculate_comprehensive_statistics(df_best, file_name)
                
                if stats:
                    solutions_stats.append(stats)
                    print(f"   ✅ Использовано {len(df_best)} решений")
                    print(f"   📊 Статистика:")
                    print(f"   ├─ Спутники: min={stats.get('ns_min', 0):.0f}, max={stats.get('ns_max', 0):.0f}, mean={stats.get('mean_ns', 0):.1f}")
                    print(f"   ├─ СКО: E={stats.get('E_std', 0):.3f}m, N={stats.get('N_std', 0):.3f}m, U={stats.get('U_std', 0):.3f}m")
                    print(f"   ├─ Ошибка высоты: max={stats.get('height_error_max', 0):.3f}m")
                    if '3d_error_mean' in stats:
                        print(f"   └─ 3D ошибка: {stats['3d_error_mean']:.3f} ± {stats['3d_error_std']:.3f} м")
            else:
                print(f"   ⚠️  Нет решений, удовлетворяющих критериям")
        else:
            print(f"   ⚠️  Нет данных после преобразования в ENU")
    
    # Сводный анализ
    print(f"\n{'='*80}")
    print("РЕЗЮМЕ АНАЛИЗА С ФИЛЬТРАЦИЕЙ ПО ENU")
    print(f"{'='*80}")
    
    # Вывод информации о примененных фильтрах
    print(f"📋 ПАРАМЕТРЫ ФИЛЬТРАЦИИ:")
    print(f"   Тип решений: {'ТОЛЬКО ФИКСИРОВАННЫЕ' if ONLY_FIXED_SOLUTIONS else 'ВСЕ РЕШЕНИЯ'}")
    print(f"   Пределы ошибок:")
    print(f"     Восток (E): ≤ {MAX_EAST_ERROR} м")
    print(f"     Север (N): ≤ {MAX_NORTH_ERROR} м") 
    print(f"     Высота (U): ≤ {MAX_HEIGHT_ERROR} м")
    print(f"     Горизонталь (2D): ≤ {MAX_HORIZONTAL_ERROR} м")
    print(f"     3D: ≤ {MAX_3D_ERROR} м")
    print(f"   Количество решений на файл: {USE_LAST_N_ROWS}")
    
    if solutions_stats:
        df_summary = pd.DataFrame(solutions_stats)
        
        print(f"\n✅ АНАЛИЗИРУЕМЫЕ ДАННЫЕ:")
        print(f"   Файлов с подходящими решениями: {len(df_summary)}")
        print(f"   Всего проанализированных решений: {df_summary['solutions_count'].sum()}")
        
        print(f"\n{'='*80}")
        print(f"СВОДНЫЙ АНАЛИЗ ТОЧНОСТИ")
        print(f"{'='*80}")
        
        # Общая статистика
        print(f"\n📊 ОБЩАЯ СТАТИСТИКА ({len(df_summary)} файлов, {df_summary['solutions_count'].sum()} решений):")
        print_metrics_summary(df_summary)
        
        # Сохранение результатов
        # timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        # solution_type = "FIXED" if ONLY_FIXED_SOLUTIONS else "ALL"
        # output_file = f"ppp_analysis_{solution_type}_ENU_filtered_{timestamp}.csv"
        # df_summary.to_csv(output_file, index=False, encoding='utf-8-sig')
        # print(f"\n💾 Отчет сохранен в {output_file}")
        
        return processed_data, solutions_stats
    
    else:
        print(f"\n⚠️  Нет решений, соответствующих заданным критериям")
        return None, None
    
import os
from pathlib import Path

def calculate_statistics_for_concatenated_data(full_filtered_data: dict) -> dict:
    """
    Рассчитывает статистику по ВСЕМ отфильтрованным данным как по единому набору.
    Аналогично calculate_comprehensive_statistics, но для склеенных данных.
    """
    if not full_filtered_data:
        return None

    # Объединяем все данные
    all_dfs = [df for df in full_filtered_data.values() if len(df) > 0]
    if not all_dfs:
        return None

    combined_df = pd.concat(all_dfs, ignore_index=True)

    # Применяем смещение ENU (если заданы) — как в оригинальной функции
    df_analysis = combined_df.copy()
    bias_applied = False
    if ENU_BIAS_E != 0.0 and 'E' in df_analysis.columns:
        df_analysis['E'] = df_analysis['E'] - ENU_BIAS_E
        bias_applied = True
    if ENU_BIAS_N != 0.0 and 'N' in df_analysis.columns:
        df_analysis['N'] = df_analysis['N'] - ENU_BIAS_N
        bias_applied = True
    if ENU_BIAS_U != 0.0 and 'U' in df_analysis.columns:
        df_analysis['U'] = df_analysis['U'] - ENU_BIAS_U
        bias_applied = True

    if bias_applied:
        print(f"   ⚙️  Применено смещение ENU для склеенных данных: "
              f"E={ENU_BIAS_E:+.4f}m, N={ENU_BIAS_N:+.4f}m, U={ENU_BIAS_U:+.4f}m")

    stats = {
        'segment': 'CONCATENATED_ALL',
        'solutions_count': len(df_analysis),
        'fix_count': (df_analysis['quality'] == 1).sum(),
        'fix_percentage': (df_analysis['quality'] == 1).sum() / len(df_analysis) * 100 if len(df_analysis) > 0 else 0,
        'mean_ns': df_analysis['ns'].mean() if 'ns' in df_analysis.columns else np.nan,
    }

    # Высотные ошибки
    if 'U' in df_analysis.columns:
        height_errors = df_analysis['U'].abs()
        stats['height_error_max'] = height_errors.max()
        stats['height_error_mean'] = height_errors.mean()
        stats['height_error_std'] = height_errors.std()
        stats['height_error_median'] = height_errors.median()

    # 3D ошибка
    if all(col in df_analysis.columns for col in ['E', 'N', 'U']):
        df_analysis['distance_3d'] = np.sqrt(df_analysis['E']**2 + df_analysis['N']**2 + df_analysis['U']**2)
        d3 = df_analysis['distance_3d']
        stats['3d_error_mean'] = d3.mean()
        stats['3d_error_std'] = d3.std()
        stats['3d_error_min'] = d3.min()
        stats['3d_error_max'] = d3.max()
        stats['3d_error_median'] = d3.median()

    # Статистика по ENU
    for component in ['E', 'N', 'U']:
        if component in df_analysis.columns:
            values = df_analysis[component].dropna()
            if len(values) > 0:
                stats[f'{component}_mean'] = values.mean()
                stats[f'{component}_std'] = values.std()
                stats[f'{component}_rms'] = np.sqrt(np.mean(values**2))
                stats[f'{component}_min'] = values.min()
                stats[f'{component}_max'] = values.max()
                stats[f'{component}_median'] = values.median()
                stats[f'{component}_mad'] = (values - values.median()).abs().median()

    # 2D и 3D ошибки
    if all(col in df_analysis.columns for col in ['E', 'N']):
        horiz = np.sqrt(df_analysis['E']**2 + df_analysis['N']**2)
        stats['2D_mean'] = horiz.mean()
        stats['2D_std'] = horiz.std()
        stats['2D_rms'] = np.sqrt(np.mean(horiz**2))
        stats['2D_median'] = horiz.median()

    if 'distance_3d' in df_analysis.columns:
        total_errors = df_analysis['distance_3d']
        stats['3D_mean'] = total_errors.mean()
        stats['3D_std'] = total_errors.std()
        stats['3D_rms'] = np.sqrt(np.mean(total_errors**2))
        stats['3D_median'] = total_errors.median()

    # Precision
    if all(col in stats for col in ['E_std', 'N_std', 'U_std']):
        stats['horizontal_precision'] = np.sqrt(stats['E_std']**2 + stats['N_std']**2)
        stats['vertical_precision'] = stats['U_std']

    return stats

def prepare_full_filtered_data(solutions: dict) -> dict:
    """
    Обрабатывает все файлы, применяет фильтры (фикс, ENU-лимиты, IQR),
    но НЕ ограничивает количество решений (игнорирует USE_LAST_N_ROWS).
    Возвращает словарь {file_name: полный отфильтрованный DataFrame}.
    """
    full_filtered = {}
    
    for file_name, df in solutions.items():
        print(f"   📊 Подготовка полных данных для графиков: {file_name}")
        
        # Преобразуем в ENU
        df_enu = reduction_to_enu(df.copy(), REF_POINT)
        if len(df_enu) == 0:
            continue
        
        # Фильтр по типу решения
        if ONLY_FIXED_SOLUTIONS:
            df_enu = filter_fixed_solutions(df_enu)
        if len(df_enu) == 0:
            continue

        # Фильтры по ENU пределам
        df_enu = filter_solutions_by_enu_limits(
            df_enu,
            max_east=MAX_EAST_ERROR,
            max_north=MAX_NORTH_ERROR,
            max_height=MAX_HEIGHT_ERROR,
            max_horizontal=MAX_HORIZONTAL_ERROR,
            max_3d=MAX_3D_ERROR
        )
        if len(df_enu) == 0:
            continue

        # Фильтр выбросов IQR
        # df_enu = filter_outliers_by_iqr(df_enu, iqr_factor=IQR_FACTOR, components=['E', 'N', 'U'])
        if len(df_enu) == 0:
            continue

        # Применяем смещение (bias) для графиков, если задано
        if ENU_BIAS_E != 0 and 'E' in df_enu:
            df_enu['E'] = df_enu['E'] - ENU_BIAS_E
        if ENU_BIAS_N != 0 and 'N' in df_enu:
            df_enu['N'] = df_enu['N'] - ENU_BIAS_N
        if ENU_BIAS_U != 0 and 'U' in df_enu:
            df_enu['U'] = df_enu['U'] - ENU_BIAS_U

        full_filtered[file_name] = df_enu

    return full_filtered

def calculate_last_n_statistics(full_filtered_: dict, last_n: int = 10) -> dict:
    """
    Рассчитывает статистику по последним `last_n` решениям из ВСЕХ отфильтрованных данных,
    сохраняя хронологический порядок (последние из последнего файла).

    Параметры:
        full_filtered_ : dict — {filename: DataFrame} после всех фильтров
        last_n : int — количество последних решений для анализа

    Возвращает:
        dict — статистика в формате, совместимом с calculate_comprehensive_statistics
    """
    if not full_filtered_ or last_n <= 0:
        return None

    # Собираем все данные в хронологическом порядке (как в склейке)
    all_dfs = []
    for file_name in sorted(full_filtered_.keys()):  # ✅ Исправлено: full_filtered_
        df = full_filtered_[file_name]               # ✅ Исправлено
        if len(df) > 0:
            all_dfs.append(df)

    if not all_dfs:
        return None

    combined = pd.concat(all_dfs, ignore_index=True)

    if len(combined) == 0:
        return None

    # Берём последние N строк
    last_n_df = combined.tail(last_n).copy()

    if len(last_n_df) == 0:
        return None

    # Применяем смещение ENU (как в оригинальной функции)
    df_analysis = last_n_df.copy()
    bias_applied = False
    if ENU_BIAS_E != 0.0 and 'E' in df_analysis.columns:
        df_analysis['E'] = df_analysis['E'] - ENU_BIAS_E
        bias_applied = True
    if ENU_BIAS_N != 0.0 and 'N' in df_analysis.columns:
        df_analysis['N'] = df_analysis['N'] - ENU_BIAS_N
        bias_applied = True
    if ENU_BIAS_U != 0.0 and 'U' in df_analysis.columns:
        df_analysis['U'] = df_analysis['U'] - ENU_BIAS_U
        bias_applied = True

    if bias_applied:
        print(f"   ⚙️  Применено смещение ENU для последних {last_n}: "
              f"E={ENU_BIAS_E:+.4f}m, N={ENU_BIAS_N:+.4f}m, U={ENU_BIAS_U:+.4f}m")

    # Расчёт статистики — копия логики из calculate_comprehensive_statistics
    stats = {
        'segment': f'LAST_{last_n}_SOLUTIONS',
        'solutions_count': len(df_analysis),
        'fix_count': (df_analysis['quality'] == 1).sum(),
        'fix_percentage': (df_analysis['quality'] == 1).sum() / len(df_analysis) * 100 if len(df_analysis) > 0 else 0,
        'mean_ns': df_analysis['ns'].mean() if 'ns' in df_analysis.columns else np.nan,
    }

    # Высота
    if 'U' in df_analysis.columns:
        height_errors = df_analysis['U'].abs()
        stats['height_error_max'] = height_errors.max()
        stats['height_error_mean'] = height_errors.mean()
        stats['height_error_std'] = height_errors.std()
        stats['height_error_median'] = height_errors.median()

    # 3D ошибка
    if all(col in df_analysis.columns for col in ['E', 'N', 'U']):
        df_analysis['distance_3d'] = np.sqrt(df_analysis['E']**2 + df_analysis['N']**2 + df_analysis['U']**2)
        d3 = df_analysis['distance_3d']
        stats['3d_error_mean'] = d3.mean()
        stats['3d_error_std'] = d3.std()
        stats['3d_error_min'] = d3.min()
        stats['3d_error_max'] = d3.max()
        stats['3d_error_median'] = d3.median()

    # ENU статистика
    for component in ['E', 'N', 'U']:
        if component in df_analysis.columns:
            values = df_analysis[component].dropna()
            if len(values) > 0:
                stats[f'{component}_mean'] = values.mean()
                stats[f'{component}_std'] = values.std()
                stats[f'{component}_rms'] = np.sqrt(np.mean(values**2))
                stats[f'{component}_min'] = values.min()
                stats[f'{component}_max'] = values.max()
                stats[f'{component}_median'] = values.median()
                stats[f'{component}_mad'] = (values - values.median()).abs().median()

    # 2D / 3D
    if all(col in df_analysis.columns for col in ['E', 'N']):
        horiz = np.sqrt(df_analysis['E']**2 + df_analysis['N']**2)
        stats['2D_mean'] = horiz.mean()
        stats['2D_std'] = horiz.std()
        stats['2D_rms'] = np.sqrt(np.mean(horiz**2))
        stats['2D_median'] = horiz.median()

    if 'distance_3d' in df_analysis.columns:
        total_errors = df_analysis['distance_3d']
        stats['3D_mean'] = total_errors.mean()
        stats['3D_std'] = total_errors.std()
        stats['3D_rms'] = np.sqrt(np.mean(total_errors**2))
        stats['3D_median'] = total_errors.median()

    # Precision
    if all(col in stats for col in ['E_std', 'N_std', 'U_std']):
        stats['horizontal_precision'] = np.sqrt(stats['E_std']**2 + stats['N_std']**2)
        stats['vertical_precision'] = stats['U_std']

    return stats

def calculate_statistics_from_segment(df_segment: pd.DataFrame, segment_name: str = "segment") -> dict:
    """
    Рассчитывает статистику для заданного сегмента данных (аналогично оригинальной функции).
    """
    if len(df_segment) == 0:
        return None

    df_analysis = df_segment.copy()

    # Применяем смещение ENU
    bias_applied = False
    if ENU_BIAS_E != 0.0 and 'E' in df_analysis.columns:
        df_analysis['E'] = df_analysis['E'] - ENU_BIAS_E
        bias_applied = True
    if ENU_BIAS_N != 0.0 and 'N' in df_analysis.columns:
        df_analysis['N'] = df_analysis['N'] - ENU_BIAS_N
        bias_applied = True
    if ENU_BIAS_U != 0.0 and 'U' in df_analysis.columns:
        df_analysis['U'] = df_analysis['U'] - ENU_BIAS_U
        bias_applied = True

    stats = {
        'segment': segment_name,
        'solutions_count': len(df_analysis),
        'fix_count': (df_analysis['quality'] == 1).sum(),
        'fix_percentage': (df_analysis['quality'] == 1).sum() / len(df_analysis) * 100 if len(df_analysis) > 0 else 0,
        'mean_ns': df_analysis['ns'].mean() if 'ns' in df_analysis.columns else np.nan,
    }

    # Высота
    if 'U' in df_analysis.columns:
        height_errors = df_analysis['U'].abs()
        stats['height_error_max'] = height_errors.max()
        stats['height_error_mean'] = height_errors.mean()
        stats['height_error_std'] = height_errors.std()
        stats['height_error_median'] = height_errors.median()

    # 3D ошибка
    if all(col in df_analysis.columns for col in ['E', 'N', 'U']):
        df_analysis['distance_3d'] = np.sqrt(df_analysis['E']**2 + df_analysis['N']**2 + df_analysis['U']**2)
        d3 = df_analysis['distance_3d']
        stats['3d_error_mean'] = d3.mean()
        stats['3d_error_std'] = d3.std()
        stats['3d_error_min'] = d3.min()
        stats['3d_error_max'] = d3.max()
        stats['3d_error_median'] = d3.median()

    # ENU
    for component in ['E', 'N', 'U']:
        if component in df_analysis.columns:
            values = df_analysis[component].dropna()
            if len(values) > 0:
                stats[f'{component}_mean'] = values.mean()
                stats[f'{component}_std'] = values.std()
                stats[f'{component}_rms'] = np.sqrt(np.mean(values**2))
                stats[f'{component}_min'] = values.min()
                stats[f'{component}_max'] = values.max()
                stats[f'{component}_median'] = values.median()

    # 2D / 3D
    if all(col in df_analysis.columns for col in ['E', 'N']):
        horiz = np.sqrt(df_analysis['E']**2 + df_analysis['N']**2)
        stats['2D_mean'] = horiz.mean()
        stats['2D_std'] = horiz.std()
        stats['2D_rms'] = np.sqrt(np.mean(horiz**2))
        stats['2D_median'] = horiz.median()

    if 'distance_3d' in df_analysis.columns:
        total_errors = df_analysis['distance_3d']
        stats['3D_mean'] = total_errors.mean()
        stats['3D_std'] = total_errors.std()
        stats['3D_rms'] = np.sqrt(np.mean(total_errors**2))
        stats['3D_median'] = total_errors.median()

    # Precision
    if all(col in stats for col in ['E_std', 'N_std', 'U_std']):
        stats['horizontal_precision'] = np.sqrt(stats['E_std']**2 + stats['N_std']**2)
        stats['vertical_precision'] = stats['U_std']

    return stats

def plot_all_diagnostics(
    full_filtered_data: dict,
    output_dir: str = "plots",
    plot_per_file: bool = True,
    plot_concatenated: bool = True,
    plot_statistics: bool = True,
    time_column: str = 'UTC'
):
    """
    Строит полный набор диагностики по отфильтрованным данным.

    Параметры:
    ----------
    full_filtered_data : dict
        Словарь {filename: DataFrame с колонками E, N, U, [UTC]}
    output_dir : str
        Папка для сохранения графиков
    plot_per_file : bool
        Строить ли отдельные графики по каждому файлу
    plot_concatenated : bool
        Строить ли склеенный график
    plot_statistics : bool
        Строить ли статистические графики (СКО, гистограммы и т.д.)
    time_column : str
        Колонка для временной оси ('UTC' или индекс)
    """
    if not full_filtered_data:
        print("⚠️ Нет данных для построения графиков")
        return

    Path(output_dir).mkdir(exist_ok=True)

    # === 1. Графики по каждому файлу ===
    if plot_per_file:
        for file_name, df in full_filtered_data.items():
            if len(df) == 0:
                continue
            time_axis = df[time_column] if time_column in df.columns and not df[time_column].isnull().all() else df.index

            fig, axs = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
            axs[0].plot(time_axis, df['E'], 'r', linewidth=0.8, alpha=0.8)
            axs[0].set_ylabel('E (м)'); axs[0].grid(True)
            axs[1].plot(time_axis, df['N'], 'g', linewidth=0.8, alpha=0.8)
            axs[1].set_ylabel('N (м)'); axs[1].grid(True)
            axs[2].plot(time_axis, df['U'], 'b', linewidth=0.8, alpha=0.8)
            axs[2].set_ylabel('U (м)'); axs[2].grid(True)
            if time_column in df.columns:
                axs[2].set_xlabel('Время')
            else:
                axs[2].set_xlabel('Индекс')
            plt.suptitle(f'ENU ошибки: {file_name}')
            plt.tight_layout()
            plt.savefig(Path(output_dir) / f"enu_{Path(file_name).stem}.png", dpi=150)
            plt.close()

    # === 2. Склеенный график с индивидуальными линиями в сеансах ===
    if plot_concatenated:
        all_E, all_N, all_U, all_time = [], [], [], []
        session_boundaries = [0]
        session_names = []
        fix_spans = []
        custom_mark_positions = []  # глобальные позиции для красных линий

        current_t = 0
        sorted_files = sorted(full_filtered_data.keys())

        for file_name in sorted_files:
            df = full_filtered_data[file_name]
            if len(df) == 0:
                current_t += 0
                continue

            df = df.copy()
            n = len(df)
            df['error_3d'] = np.sqrt(df['E']**2 + df['N']**2 + df['U']**2)

            # --- 🔴 Индивидуальная метка для этого сеанса ---
            mark_local = None
            if file_name in SESSION_MARK_CONFIG:
                spec = SESSION_MARK_CONFIG[file_name]

                if spec is None:
                    mark_local = None
                elif spec == "end":
                    mark_local = n - 1
                elif spec == "start":
                    mark_local = 0
                elif spec == "last_fix":
                    fix_idx = df[df['quality'] == 1].index
                    if len(fix_idx) > 0:
                        mark_local = fix_idx[-1] - df.index[0]  # локальный индекс
                elif isinstance(spec, (int, float)):
                    if 0 <= spec <= 1:
                        # Доля от длины сеанса
                        mark_local = int(spec * (n - 1))
                    elif spec >= 1:
                        # Абсолютная локальная позиция (не глобальная!)
                        mark_local = int(spec)
                        if mark_local >= n:
                            mark_local = n - 1  # ограничить концом
                # else: неизвестный тип — игнорируем

                # Сохраняем глобальную позицию
                if mark_local is not None and 0 <= mark_local < n:
                    custom_mark_positions.append(current_t + mark_local)

            # --- Устойчивый FIX (опционально) ---
            fix_start_local = None
            for i in range(n - 1, -1, -1):
                if df['quality'].iloc[i] == 1 and df['error_3d'].iloc[i] <= 0.1:
                    fix_start_local = i
                else:
                    break
            if fix_start_local is not None and fix_start_local < n - 1:
                fix_spans.append((current_t + fix_start_local, current_t + n - 1))

            # --- Данные ---
            segment_time = np.arange(current_t, current_t + n)
            all_time.extend(segment_time)
            all_E.extend(df['E'].values)
            all_N.extend(df['N'].values)
            all_U.extend(df['U'].values)

            # --- Границы ---
            if current_t > 0:
                session_boundaries.append(current_t)
                session_names.append(Path(file_name).stem)

            current_t += n

        total_points = len(all_time)
        if total_points == 0:
            return

        fig, axs = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

        # Основные графики
        axs[0].plot(all_time, all_E, 'r', linewidth=0.6, alpha=0.7)
        axs[0].set_ylabel('E (м)'); axs[0].grid(True)
        axs[1].plot(all_time, all_N, 'g', linewidth=0.6, alpha=0.7)
        axs[1].set_ylabel('N (м)'); axs[1].grid(True)
        axs[2].plot(all_time, all_U, 'b', linewidth=0.6, alpha=0.7)
        axs[2].set_ylabel('U (м)'); axs[2].grid(True)

        # --- Заливка FIX ---
        for start, end in fix_spans:
            for ax in axs:
                ax.axvspan(start, end, color='lime', alpha=0.3)
            axs[2].text((start + end) / 2, axs[2].get_ylim()[1] * 0.9, 'FIX',
                        fontsize=10, color='darkgreen', weight='bold', ha='center')

        # --- Границы сеансов ---
        for pos in session_boundaries:
            for ax in axs:
                ax.axvline(x=pos, color='k', linestyle='--', linewidth=0.8, alpha=0.6)

        # --- 🔴 Индивидуальные красные линии + время от начала сеанса ---
        if SESSION_MARK_CONFIG:
            current_t = 0
            all_utc = []  # для fallback, если нужно

            for file_name in sorted_files:
                df = full_filtered_data[file_name]
                if len(df) == 0:
                    continue

                n = len(df)
                has_utc = 'UTC' in df.columns and not df['UTC'].isnull().all()

                # Определяем локальную метку (как раньше)
                mark_local = None
                if file_name in SESSION_MARK_CONFIG:
                    spec = SESSION_MARK_CONFIG[file_name]
                    if spec == "end":
                        mark_local = n - 1
                    elif spec == "start":
                        mark_local = 0
                    elif spec == "last_fix":
                        fix_idx = df[df['quality'] == 1].index
                        if len(fix_idx) > 0:
                            mark_local = fix_idx[-1] - df.index[0]
                    elif isinstance(spec, (int, float)):
                        if 0 <= spec <= 1:
                            mark_local = int(spec * (n - 1))
                        elif spec >= 1:
                            mark_local = min(int(spec), n - 1)

                # Если метка задана — рисуем
                if mark_local is not None and 0 <= mark_local < n:
                    global_pos = current_t + mark_local

                    # --- Вычисление локального времени ---
                    if has_utc:
                        session_start_time = df['UTC'].iloc[0]
                        mark_time = df['UTC'].iloc[mark_local]
                        delta_sec = (mark_time - session_start_time).total_seconds()
                    else:
                        # Эмуляция: 1 строка = 1 секунда
                        delta_sec = mark_local  # так как начинаем с 0

                    delta_min = delta_sec / 60.0

                    # --- Рисуем линию на всех трёх графиках ---
                    for ax in axs:
                        ax.axvline(x=global_pos, color='red', linestyle='-', linewidth=1.3, alpha=0.85)

                    # --- Надпись на нижнем графике ---
                    axs[2].text(
                        global_pos,
                        axs[2].get_ylim()[1],
                        f"{delta_min:.1f} мин",
                        color='red',
                        fontsize=9,
                        verticalalignment='bottom',
                        horizontalalignment='center',
                        weight='bold'
                    )

                current_t += n

        plt.suptitle('Совмещенные ENU ошибки по каждому решению (GPS+Galileo, PPP-AR)\n')
        plt.tight_layout()
        plt.savefig(Path(output_dir) / "enu_concatenated_full.png", dpi=150, bbox_inches='tight')
        plt.close()

    # === 3. Статистические графики ===
    if plot_statistics:
        # Собираем все данные в один DataFrame
        all_dfs = [df[['E', 'N', 'U']] for df in full_filtered_data.values() if len(df) > 0]
        if not all_dfs:
            return
        combined = pd.concat(all_dfs, ignore_index=True)

        # 3.1 Boxplot СКО (на основе отклонений от среднего в каждом файле)
        sko_data = {'E': [], 'N': [], 'U': []}
        for df in full_filtered_data.values():
            if len(df) > 1:
                sko_data['E'].append(df['E'].std())
                sko_data['N'].append(df['N'].std())
                sko_data['U'].append(df['U'].std())
        if any(sko_data.values()):
            plt.figure(figsize=(8, 5))
            plt.boxplot([sko_data['E'], sko_data['N'], sko_data['U']], labels=['E', 'N', 'U'])
            plt.title('СКО (σ) по компонентам — по файлам')
            plt.ylabel('СКО (м)')
            plt.grid(True, axis='y')
            plt.savefig(Path(output_dir) / "sko_per_file.png", dpi=150)
            plt.close()

        # 3.2 Гистограммы распределения ошибок
        fig, axs = plt.subplots(1, 3, figsize=(15, 4))
        colors = ['red', 'green', 'blue']
        for i, comp in enumerate(['E', 'N', 'U']):
            axs[i].hist(combined[comp].dropna(), bins=40, color=colors[i], alpha=0.7)
            axs[i].set_title(f'{comp}: μ={combined[comp].mean():.4f}, σ={combined[comp].std():.4f}')
            axs[i].grid(True)
        plt.tight_layout()
        plt.savefig(Path(output_dir) / "enu_histograms_full.png", dpi=150)
        plt.close()

        # 3.3 Scatter E-N
        plt.figure(figsize=(7, 7))
        plt.scatter(combined['N'], combined['E'], s=1, alpha=0.5, c='darkgreen')
        plt.xlabel('North (м)'); plt.ylabel('East (м)')
        plt.title('Горизонтальное распределение ошибок (все данные)')
        plt.axis('equal'); plt.grid(True)
        plt.savefig(Path(output_dir) / "en_scatter_full.png", dpi=150)
        plt.close()

        # 3.4 Корреляция
        corr = combined[['E', 'N', 'U']].corr()
        plt.figure(figsize=(6, 5))
        sns.heatmap(corr, annot=True, fmt=".3f", cmap='coolwarm', vmin=-1, vmax=1)
        plt.title('Корреляция ENU ошибок')
        plt.savefig(Path(output_dir) / "enu_correlation_full.png", dpi=150)
        plt.close()

        # 3.5 2D и 3D ошибки во времени (по склейке)
        if plot_concatenated and all_time:
            horiz = np.sqrt(np.array(all_E)**2 + np.array(all_N)**2)
            three_d = np.sqrt(np.array(all_E)**2 + np.array(all_N)**2 + np.array(all_U)**2)

            plt.figure(figsize=(12, 5))
            plt.plot(all_time, horiz, label='2D ошибка', color='orange', linewidth=0.8)
            plt.plot(all_time, three_d, label='3D ошибка', color='purple', linewidth=0.8)
            plt.xlabel('Номер измерения'); plt.ylabel('Ошибка (м)')
            plt.title('2D и 3D ошибки (все данные)')
            plt.legend(); plt.grid(True)
            plt.savefig(Path(output_dir) / "2d3d_errors_full.png", dpi=150)
            plt.close()

    # === 4. Статистика по склеенным данным ===
    concat_stats = calculate_statistics_for_concatenated_data(full_filtered_data)
    if concat_stats:
        print(f"\n{'='*80}")
        print("СТАТИСТИКА ПО СКЛЕЕННЫМ ДАННЫМ (все отфильтрованные решения)")
        print(f"{'='*80}")
        print(f"   📏 Всего решений: {concat_stats['solutions_count']}")
        print(f"   ✅ Фиксированных: {concat_stats['fix_count']} ({concat_stats['fix_percentage']:.2f}%)")
        print(f"   📊 СКО (σ): E={concat_stats.get('E_std', 0):.4f} м, "
                f"N={concat_stats.get('N_std', 0):.4f} м, U={concat_stats.get('U_std', 0):.4f} м")
        print(f"   🎯 3D RMS: {concat_stats.get('3D_rms', 0):.4f} м")
        print(f"   📐 Горизонтальная точность: {concat_stats.get('horizontal_precision', 0):.4f} м")
        print(f"   📏 Вертикальная точность (СКО): {concat_stats.get('vertical_precision', 0):.4f} м")
        if 'height_error_max' in concat_stats:
            print(f"   📈 Макс. ошибка высоты: {concat_stats['height_error_max']:.4f} м")

    # === 5. Статистика по последним N решениям ===
    if ENABLE_LAST_N_STATS and STAT_LAST_N > 0:
        last_stats = calculate_last_n_statistics(full_filtered_data, last_n=STAT_LAST_N)
        if last_stats:
            print(f"\n{'='*80}")
            print(f"СТАТИСТИКА ПО ПОСЛЕДНИМ {STAT_LAST_N} РЕШЕНИЯМ (все файлы, хронология сохранена)")
            print(f"{'='*80}")
            print(f"   📏 Решений: {last_stats['solutions_count']}")
            print(f"   ✅ Фикс: {last_stats['fix_count']} ({last_stats['fix_percentage']:.2f}%)")
            print(f"   📊 СКО: E={last_stats.get('E_std', 0):.4f} м, "
                    f"N={last_stats.get('N_std', 0):.4f} м, U={last_stats.get('U_std', 0):.4f} м")
            print(f"   🎯 3D RMS: {last_stats.get('3D_rms', 0):.4f} м")
            print(f"   📐 Горизонтальная точность: {last_stats.get('horizontal_precision', 0):.4f} м")
            print(f"   📏 Вертикальная точность (СКО): {last_stats.get('vertical_precision', 0):.4f} м")
            if 'height_error_max' in last_stats:
                print(f"   📈 Макс. ошибка высоты: {last_stats['height_error_max']:.4f} м")

    print(f"✅ Диагностические графики сохранены в '{output_dir}'")

# Обновлённая основная функция с графиками
def improved_main_analysis_with_plots():
    """Основной анализ + графики по полным данным"""
    print("📁 Загрузка POS-файлов...")
    solutions = {}
    pos_files = sorted(glob.glob(os.path.join(SOLUTIONS_DIR, "*.pos")))
    
    for file_path in pos_files:
        file_name = os.path.basename(file_path)
        df = read_pos_file_corrected(file_path)
        if len(df) > 0:
            solutions[file_name] = df

    if not solutions:
        print("❌ Не найдено POS-файлов")
        return

    # Анализ с отбором лучших (для статистики)
    processed_data, statistics = improved_main_analysis()  # ваша существующая функция

    # Подготовка ПОЛНЫХ данных для графиков
    print("\n🎨 Подготовка полных данных для графиков (без ограничения USE_LAST_N_ROWS)...")
    full_filtered = prepare_full_filtered_data(solutions)

    # Построение графиков
    plot_all_diagnostics(
        full_filtered_data=full_filtered,
        output_dir="plots",
        plot_per_file=True,
        plot_concatenated=True,
        plot_statistics=True,
        time_column='UTC'
    )

    # === Статистика от метки до конца сеанса ===
    if ENABLE_STATS_FROM_MARK and SESSION_MARK_CONFIG:
        print(f"\n{'='*80}")
        print("СТАТИСТИКА ОТ МЕТКИ ДО КОНЦА СЕАНСА")
        print(f"{'='*80}")

        sorted_files = sorted(full_filtered.keys())
        all_mark_stats = []

        for file_name in sorted_files:
            if file_name not in SESSION_MARK_CONFIG or file_name not in full_filtered:
                continue

            df = full_filtered[file_name].copy()
            if len(df) == 0:
                continue

            n = len(df)
            spec = SESSION_MARK_CONFIG[file_name]

            # Определяем локальную позицию метки (как при рисовании)
            mark_local = None
            if spec == "end":
                mark_local = n - 1
            elif spec == "start":
                mark_local = 0
            elif spec == "last_fix":
                fix_idx = df[df['quality'] == 1].index
                if len(fix_idx) > 0:
                    mark_local = fix_idx[-1] - df.index[0]
            elif isinstance(spec, (int, float)):
                if 0 <= spec <= 1:
                    mark_local = int(spec * (n - 1))
                elif spec >= 1:
                    mark_local = min(int(spec), n - 1)

            if mark_local is None or not (0 <= mark_local < n):
                continue

            # Берём данные от метки до конца
            df_from_mark = df.iloc[mark_local:].copy()

            # Считаем статистику
            stats = calculate_statistics_from_segment(df_from_mark, segment_name=file_name)
            if stats and stats['solutions_count'] > 0:
                all_mark_stats.append(stats)

                print(f"\n📁 Сеанс: {file_name}")
                print(f"   📏 Решений после метки: {stats['solutions_count']}")
                print(f"   ✅ Фикс: {stats['fix_count']} ({stats['fix_percentage']:.2f}%)")
                print(f"   📊 СКО: E={stats.get('E_std', 0):.4f} м, "
                      f"N={stats.get('N_std', 0):.4f} м, U={stats.get('U_std', 0):.4f} м")
                print(f"   🎯 3D RMS: {stats.get('3D_rms', 0):.4f} м")
                print(f"   📐 Горизонтальная точность: {stats.get('horizontal_precision', 0):.4f} м")

        # Опционально: сводка по всем сеансам
        if all_mark_stats:
            df_summary = pd.DataFrame(all_mark_stats)
            print(f"\n{'='*60}")
            print("СВОДКА ПО ВСЕМ СЕАНСАМ (от метки до конца)")
            print(f"{'='*60}")
            print(f"   📊 Среднее СКО: E={df_summary['E_std'].mean():.4f} ± {df_summary['E_std'].std():.4f} м")
            print(f"   📊 Среднее СКО: N={df_summary['N_std'].mean():.4f} ± {df_summary['N_std'].std():.4f} м")
            print(f"   📊 Среднее СКО: U={df_summary['U_std'].mean():.4f} ± {df_summary['U_std'].std():.4f} м")
            print(f"   🎯 Средний 3D RMS: {df_summary['3D_rms'].mean():.4f} ± {df_summary['3D_rms'].std():.4f} м")

def print_metrics_summary(df):
    """Вывод сводки метрик для DataFrame"""
    
    sample_sizes = df['solutions_count']
    
    print(f"   📏 Размер выборки: {sample_sizes.mean():.1f} ± {sample_sizes.std():.1f} решений на файл")
    print(f"   ✅ Фикс: {df['fix_percentage'].mean():.1f} ± {df['fix_percentage'].std():.1f}%")
    
    metric_groups = [
        ("СКО", [
            ('E_std', 'E (м)'),
            ('N_std', 'N (м)'), 
            ('U_std', 'U (м)'),
        ]),
        ("СРЕДНЕЕ", [
            ('E_mean', 'E (м)'),
            ('N_mean', 'N (м)'),
            ('U_mean', 'U (м)'),
        ]),
        ("СКП", [
            ('E_rms', 'E (м)'),
            ('N_rms', 'N (м)'),
            ('U_rms', 'U (м)'),
        ])
    ]
    
    for group_name, metrics in metric_groups:
        print(f"   📈 {group_name}:")
        for col, desc in metrics:
            if col in df.columns:
                values = df[col].dropna()
                if len(values) > 0:
                    mean_val = values.mean()
                    std_val = values.std()
                    print(f"     {desc:8} {mean_val:7.4f} ± {std_val:6.4f} м")
    
    # 2D и 3D метрики
    if '2D_rms' in df.columns:
        print(f"   🎯 2D СКП: {df['2D_rms'].mean():.4f} ± {df['2D_rms'].std():.4f} м")
    if '3D_rms' in df.columns:
        print(f"   🎯 3D СКП: {df['3D_rms'].mean():.4f} ± {df['3D_rms'].std():.4f} м")
    
    # Статистика высоты
    if 'height_error_max' in df.columns:
        print(f"   📊 Ошибка высоты: max={df['height_error_max'].max():.4f} м, mean={df['height_error_mean'].mean():.4f} м")

# Запуск улучшенного анализа
if __name__ == "__main__":
    improved_main_analysis_with_plots()

📁 Загрузка POS-файлов...
   📖 Чтение всего файла: sol_000.pos
✅ Загружено: 8972 записей (фикс: 0, 0.0%)
   📖 Чтение всего файла: sol_001.pos
✅ Загружено: 8943 записей (фикс: 0, 0.0%)
   📖 Чтение всего файла: sol_002.pos
✅ Загружено: 8929 записей (фикс: 0, 0.0%)
   📖 Чтение всего файла: sol_003.pos
✅ Загружено: 8973 записей (фикс: 0, 0.0%)
   📖 Чтение всего файла: sol_004.pos
✅ Загружено: 8953 записей (фикс: 0, 0.0%)
   📖 Чтение всего файла: sol_005.pos
✅ Загружено: 3518 записей (фикс: 0, 0.0%)
   📖 Чтение всего файла: sol_006.pos
   ⚠️  Файл solutions_all_AR/sol_006.pos пуст после фильтрации
   📖 Чтение всего файла: sol_007.pos
   ⚠️  Файл solutions_all_AR/sol_007.pos пуст после фильтрации
📁 Загрузка POS-файлов...
   📖 Чтение всего файла: sol_000.pos
✅ Загружено: 8972 записей (фикс: 0, 0.0%)
   📖 Чтение всего файла: sol_001.pos
✅ Загружено: 8943 записей (фикс: 0, 0.0%)
   📖 Чтение всего файла: sol_002.pos
✅ Загружено: 8929 записей (фикс: 0, 0.0%)
   📖 Чтение всего файла: sol_003.pos
✅ 